# NONAN source-compatibility gate

Assess the 80 healthy candidates against existing healthy development sources. This is participant-level, healthy-only, and does not train or tune the stroke classifier; frozen NONAN and RevalExo are not read.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT; P=ROOT/'data'/'processed'; N=ROOT/'data'/'interim'/'nonan_gaitprint'
parts=pd.read_csv(N/'participant_partitions.csv'); candidate=set(parts.loc[parts.partition.eq('candidate_healthy_enrichment'),'participant_id']); forbidden=set(parts.loc[~parts.partition.eq('candidate_healthy_enrichment'),'participant_id']); nm=pd.read_csv(N/'candidate_healthy_enrichment_window_metadata.csv'); assert set(nm.participant)==candidate and not(set(nm.participant)&forbidden)
def feats(x,m,key,src):
 rows=[]
 for pid,idx in m.groupby(key).indices.items():
  a=np.asarray(x[np.asarray(list(idx))],dtype='float32'); f=a.reshape(-1,3); d=np.diff(a,axis=1).reshape(-1,3); vals=[np.median(f,0),np.percentile(f,75,0)-np.percentile(f,25,0),np.percentile(f,95,0),np.sqrt(np.mean(d*d,0))]; rows.append({'participant_key':str(pid),'source':src,**{f'{s}_{c}':float(v) for s,z in zip(['median','iqr','p95','diff_rms'],vals) for c,v in zip(['LB','LF','RF'],z)}})
 return pd.DataFrame(rows)
base=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); bm=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); keep=bm.label.eq('healthy').to_numpy(); base,bm=base[keep],bm.loc[keep].reset_index(drop=True)
ref=pd.concat([feats(base[bm.dataset_id.eq(s).to_numpy()],bm.loc[bm.dataset_id.eq(s)].reset_index(drop=True),'participant_key',s) for s in sorted(bm.dataset_id.unique())],ignore_index=True); cand=feats(np.load(N/'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy',mmap_mode='r'),nm,'participant','nonan_gaitprint'); z=pd.concat([ref,cand],ignore_index=True); cols=[c for c in z if c not in {'participant_key','source'}]; y=z.source.eq('nonan_gaitprint').astype(int).to_numpy(); model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced',random_state=20260902)); prob=cross_val_predict(model,z[cols],y,cv=StratifiedKFold(5,shuffle=True,random_state=20260902),method='predict_proba')[:,1]; out={'participants':len(z),'nonan':int(y.sum()),'reference_healthy':int((1-y).sum()),'source_auroc':float(roc_auc_score(y,prob))}; print(out); pd.DataFrame({'participant_key':z.participant_key,'source':z.source,'nonan_probability':prob}).to_csv(N/'candidate_healthy_enrichment_source_compatibility_predictions.csv',index=False); (N/'candidate_healthy_enrichment_source_compatibility_summary.json').write_text(json.dumps(out,indent=2),encoding='utf-8')

{'participants': 206, 'nonan': 80, 'reference_healthy': 126, 'source_auroc': 0.8925595238095239}


106

A high source-AUROC is a domain-shift warning, not an automatic rejection. The next experiment, if justified, remains bounded source-aware enrichment with unchanged frozen cohorts.